# 01_eda: Exploratory Data Analysis
Understand the loan data, find leakage columns, and check class imbalance (REQ-01).

In [0]:
%run ./00_config

# Part A: Data Source Selection
Before modeling, we check whether the suggested file (`LoanStats_2018Q2.csv`) is fit for training a default model.

## Step 1: List files in the data folder
The brief gives us a folder path but not the file names. Before writing any code against the data,
we list what is actually inside the folder. This way we build on facts, not assumptions.

In [0]:
# WHAT: List every file inside the Lending Club sample folder.
# WHY:  The brief only gives the folder path. We need the real file names
#       (and their sizes) before we can read the data correctly.

files = dbutils.fs.ls(RAW_DIR)
display(files)

**Result:** The folder contains 2 files:
- `LoanStats_2018Q2.csv` (~67 MB): the raw loan records we will use.
- `lendingClubData.txt` (~1 KB): a short description file.

**Why it matters:** The file name says *2018 Q2*, so the data may only cover loans issued in April–June 2018.
If so, many loans may still be "Current" (outcome not known yet), which will affect how we build the target
and how we split train/test by time. We will confirm this with real numbers when we load the data.

## Step 2: Read the dataset's description file
Before loading the 67 MB CSV, we read the small description file that ships with it.
It may tell us the data's source, time period, or known quirks.

In [0]:
# WHAT: Print the contents of the small description file.
# WHY:  It may explain where the data comes from and which time period it covers.
#       That context guides how we read and split the data later.

print(dbutils.fs.head(RAW_DIR + "lendingClubData.txt"))

**Result:** The description says the data comes from LendingClub's public loan exports and includes
each loan's *current status* (Current, Late, Fully Paid, etc.) and its *latest payment information*.

**Why it matters:**
1. `loan_status` has more than two values, so we must decide how to turn it into a yes/no target (some loans are still "Current").
2. "Latest payment information" means some columns describe what happened **after** the loan was issued.
   These are **data leakage** risks and must be found and removed in EDA.
3. It is not yet clear whether the file covers only 2018 Q2 or all loans up to 2018 Q2. We will check `issue_d`.

## Step 3: Peek at the raw CSV text
Real-world CSV exports often contain extra lines (titles, notes) above the header or below the data.
We look at the first characters of the raw file so we know exactly how to read it into Spark.

In [0]:
# WHAT: Print the first 1,500 characters of the raw loan CSV as plain text.
# WHY:  To check the file's real structure (is line 1 the header? any extra title lines?)
#       before telling Spark how to read it.

print(dbutils.fs.head(RAW_CSV_PATH, 1500))

Line 1 is the column names, so we can read the file with `header=True`.

## Step 4: Load the data and count rows and columns

In [0]:

# Read the CSV into a Spark DataFrame
raw_df = (spark.read
          .option("header", True)       # line 1 = column names (checked in Step 3)
          .option("inferSchema", True)  # let Spark guess each column's type
          .csv(RAW_CSV_PATH))

# Count rows and columns. This is our starting size.
print(f"Rows:    {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns)}")

Loaded **130,776 rows** and **145 columns**. We will only keep the useful, safe columns.

## Step 5: Count the column data types

In [0]:
# Count how many columns Spark read as each type (string, int, double...)
# Models need numbers, so text columns will need fixing later
from collections import Counter

type_counts = Counter(col_type for col_name, col_type in raw_df.dtypes)
print(type_counts)

Column types: **80 int**, **24 double**, **41 string**.
The 41 text columns need checking: some are real categories (like `purpose`), others are numbers stored as text.

## Step 6: List the text (string) columns


In [0]:
# List the names of all text columns, so we can sort them:
# real categories, numbers stored as text, or columns to drop
string_cols = [col_name for col_name, col_type in raw_df.dtypes if col_type == "string"]

print(f"{len(string_cols)} text columns:")
print(string_cols)

**Sorting the 41 text columns:**

| Group | Columns | Plan |
|---|---|---|
| IDs / links | `id`, `member_id`, `url` | Drop: no predictive meaning |
| Free text | `emp_title`, `desc`, `title` | Drop: too messy (thousands of spellings) |
| Numbers stored as text | `term`, `int_rate`, `revol_util`, `emp_length` | Clean and convert in feature engineering |
| Categories | `grade`, `sub_grade`, `home_ownership`, `verification_status`, `purpose`, `addr_state`, `zip_code`, `initial_list_status`, `application_type`, `verification_status_joint`, `disbursement_method` | Encode in feature engineering |
| Dates | `issue_d`, `earliest_cr_line`, `sec_app_earliest_cr_line` | `issue_d` for the time split; credit line dates → credit history length |
| Target | `loan_status` | Turn into 0/1 |
| **Leakage** ⚠️ | `pymnt_plan`, `last_pymnt_d`, `next_pymnt_d`, `last_credit_pull_d`, all `hardship_*`, `payment_plan_start_date`, `debt_settlement_flag`, `debt_settlement_flag_date`, `settlement_status`, `settlement_date` | **Drop**: created *after* the loan starts, so not known on application day |

## Step 7: List the number columns (int and double)

In [0]:
# List all number columns (everything that is NOT text)
# Leakage can hide here too, e.g. total_pymnt (amount paid after the loan started)
numeric_cols = [col_name for col_name, col_type in raw_df.dtypes if col_type != "string"]

print(f"{len(numeric_cols)} number columns:")
print(numeric_cols)

**104 number columns** (80 int + 24 double). Leakage check, using the test *"Would we know this on application day?"*:

| Group | Columns | Why it leaks |
|---|---|---|
| Payments during the loan | `out_prncp`, `out_prncp_inv`, `total_pymnt`, `total_pymnt_inv`, `total_rec_prncp`, `total_rec_int`, `total_rec_late_fee`, `last_pymnt_amnt` | Created month by month *after* the loan starts |
| Recovery after default | `recoveries`, `collection_recovery_fee` | Only exist *after* the loan has already gone bad |
| Hardship / settlement | `deferral_term`, `hardship_*` (6 columns), `orig_projected_additional_accrued_interest`, `settlement_amount`, `settlement_percentage`, `settlement_term` | Only appear when the borrower is *already* struggling to pay |

All other number columns describe the applicant's credit history **before** applying, so they are safe.

## Step 8: Check the leakage list against the data

In [0]:
# Check that every leakage column name really exists in the data
# (a typo would mean the column is NOT dropped, and leakage would stay hidden)
missing = [c for c in LEAKAGE_COLS if c not in raw_df.columns]

print(f"Leakage columns listed: {len(LEAKAGE_COLS)}")
print(f"Names not found in data: {missing}")

All **29 leakage columns** in `LEAKAGE_COLS` exist in the data (no typos). The list is stored once in `00_config`,
so feature engineering can drop them all with one line. (The 2018 Q2 file also had 7 settlement columns,
but they do not exist in the main dataset, so they were removed from the list; see Step 18.)

## Step 9: Investigate why `id` was read as text

In [0]:
# Why is 'id' text? Look at the rows where id actually has a value
from pyspark.sql import functions as F

id_rows = raw_df.filter(F.col("id").isNotNull())

print(f"Rows with an id value: {id_rows.count():,} out of {raw_df.count():,}")
display(id_rows.select("id", "loan_amnt", "loan_status").limit(10))

Only **2 rows** have an `id`, and they are **summary lines** at the end of the file
("Total amount funded in policy code 1/2"), not loans. Their other columns are empty.
All real loans have an empty `id` (LendingClub removed IDs for privacy).
**Action:** remove these 2 rows → **130,774 real loans**.

## Step 10: Remove the 2 summary rows (keep real loans only)

In [0]:
# Keep only real loans: every real loan has a loan amount,
# the 2 summary lines at the end of the file do not
loans_df = raw_df.filter(F.col("loan_amnt").isNotNull())

print(f"Before: {raw_df.count():,} rows")
print(f"After:  {loans_df.count():,} rows")
print(f"Removed: {raw_df.count() - loans_df.count():,} rows")

In [0]:
# 4 rows were removed, not 2. Let's see ALL rows that have no loan amount
no_amount = raw_df.filter(F.col("loan_amnt").isNull())

display(no_amount.select("id", "loan_amnt", "loan_status", "grade", "int_rate", "annual_inc"))

Step 10 removed **4 rows**, not 2. Checking them (Step 10b) showed:
- 2 rows are **completely empty** (blank lines near the end of the file)
- 2 rows are the **summary lines** found in Step 9

None are real loans, so removing them is correct. **Real loans: 130,772.**

## Step 11: Save the real loans as a Delta table

In [0]:
# Save the 130,772 real loans from the 2018 Q2 file as a Delta table
# (its own table, so it never mixes with the main Parquet data)
(loans_df.write
    .format("delta")        # Delta Lake format (fast, reliable, keeps history)
    .mode("overwrite")      # replace the table if it exists, so re-running is safe
    .saveAsTable(CSV_2018Q2_TABLE))

print(f"Saved {spark.table(CSV_2018Q2_TABLE).count():,} rows to {CSV_2018Q2_TABLE}")

Saved **130,772 real loans** to `workspace.loan_default.loans_raw` (Delta).
Later notebooks read this table directly: faster, and the junk rows are already removed.

## Step 12: Distribution of the target (`loan_status`)

In [0]:
# Count loans per status, with the % of all loans
# This shows the target's real values and how imbalanced they are
display(spark.sql(f"""
    SELECT loan_status,
           COUNT(*) AS loans,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {CSV_2018Q2_TABLE}
    GROUP BY loan_status
    ORDER BY loans DESC
"""))

**93.65%** of loans are still **"Current"** (outcome not known yet), because they were issued in 2018 Q2 and the file was saved soon after.
Only **5,794** loans have a final outcome (**5,660 Fully Paid**, **134 Charged Off**), so the default rate among finished loans is about **2.3%**.
Early "Fully Paid" loans are early payoffs, so they are not typical borrowers. This file may not be enough to train a reliable model.

In [0]:
# Does Databricks have another Lending Club sample with older, finished loans?
display(dbutils.fs.ls("/databricks-datasets/samples/lending_club/"))

In [0]:
# Read the description of the second Lending Club sample before using it
print(dbutils.fs.head("/databricks-datasets/samples/lending_club/readme.md"))

In [0]:
# Load the second Lending Club sample (Parquet) and check its size and target values
ALT_PATH = "/databricks-datasets/samples/lending_club/parquet/"

alt_df = spark.read.parquet(ALT_PATH)   # Parquet stores types, so no inferSchema needed

print(f"Rows:    {alt_df.count():,}")
print(f"Columns: {len(alt_df.columns)}")

# Count loans per status, biggest first
display(alt_df.groupBy("loan_status").count().orderBy(F.desc("count")))

In [0]:
# Which years do these loans come from? Count loans per issue year
display(alt_df
        .withColumn("issue_year", F.substring("issue_d", -4, 4))
        .groupBy("issue_year").count()
        .orderBy("issue_year"))

## Step 13: Look for a dataset with finished loans

In [0]:
# Does Databricks have another Lending Club sample with older, finished loans?
display(dbutils.fs.ls("/databricks-datasets/samples/lending_club/"))

In [0]:
# Read the description of the second Lending Club sample before using it
print(dbutils.fs.head("/databricks-datasets/samples/lending_club/readme.md"))

Found a second built-in sample: the Kaggle Lending Club dataset saved as Parquet by Databricks (CC BY 4.0).

## Step 14: Check the second dataset's size and target

In [0]:
# Load the Parquet dataset and check its size and target values
alt_df = spark.read.parquet(LOANS_PARQUET_PATH)   # Parquet stores types, so no inferSchema needed

print(f"Rows:    {alt_df.count():,}")
print(f"Columns: {len(alt_df.columns)}")

# Count loans per status, biggest first
display(alt_df.groupBy("loan_status").count().orderBy(F.desc("count")))

**1,481,560 loans** and **137 columns**. **658,049** have a final outcome (522,325 Fully Paid + 135,724 Charged Off/Default),
about **113× more** than the 2018 Q2 file, with a default rate of about **20.6%** among finished loans.
A few broken rows exist (status `null` or a date like `Oct-2015`), to be removed during cleaning.

## Step 15: Which years do the loans cover?

In [0]:
# Which years do these loans come from? Count loans per issue year
display(alt_df
        .withColumn("issue_year", F.substring("issue_d", -4, 4))
        .groupBy("issue_year").count()
        .orderBy("issue_year"))

Loans were issued from **2012 to 2017**, which allows a realistic time-based train/test split.

## ✅ Decision: use the 2012–2017 Parquet dataset

| | 2018 Q2 CSV (suggested) | 2012–2017 Parquet (chosen) |
|---|---|---|
| Loans with a final outcome | 5,794 | 658,049 |
| Defaults | 134 | 135,724 |
| Default rate (finished loans) | 2.3% | ~20.6% |
| Years covered | 2018 Q2 only | 2012–2017 |

The suggested file is **93.65% "Current"** loans with no outcome yet, and its few finished loans are mostly early payoffs.
The Parquet dataset has real outcomes, enough defaults to learn from, and multiple years for a time-based split.
**All following steps use the Parquet dataset.**

# Part B: Main Dataset (Lending Club 2012–2017)
All following steps use the Parquet dataset chosen in Part A.

## Step 16: Keep only rows with a valid loan status

In [0]:
# Load the main dataset
main_df = spark.read.parquet(LOANS_PARQUET_PATH)

# The 7 real loan statuses ("guest list"); anything else is a broken row
VALID_STATUSES = ["Current", "Fully Paid", "Charged Off", "Late (31-120 days)",
                  "In Grace Period", "Late (16-30 days)", "Default"]

main_loans_df = main_df.filter(F.col("loan_status").isin(VALID_STATUSES))

print(f"Before:  {main_df.count():,} rows")
print(f"After:   {main_loans_df.count():,} rows")
print(f"Removed: {main_df.count() - main_loans_df.count():,} rows")

Removed **19 broken rows** (18 with no status + 1 with a date in the status column).
**1,481,541 valid loans** remain.

## Step 17: Save the valid loans as the main Delta table

In [0]:
# Save the 1,481,541 valid loans as our main Delta table
# Later notebooks read this table instead of the Parquet files
(main_loans_df.write
    .format("delta")                      # Delta Lake format (fast, reliable, keeps history)
    .mode("overwrite")                    # replace the rows if the table exists
    .option("overwriteSchema", "true")    # also replace the column types (old table had CSV types)
    .saveAsTable(RAW_TABLE))

print(f"Saved {spark.table(RAW_TABLE).count():,} rows to {RAW_TABLE}")

Saved **1,481,541 valid loans** to `workspace.loan_default.loans_raw` (Delta).
`overwriteSchema` was needed because the table previously held the 2018 Q2 CSV data, which had different column types.

## Step 18: Compare columns with the 2018 Q2 file and re-check leakage

In [0]:
# Compare the main table's columns with the 2018 Q2 file's columns
main_cols = spark.table(RAW_TABLE).columns

missing_leak = [c for c in LEAKAGE_COLS if c not in main_cols]   # A: leakage names not in new data
new_cols     = [c for c in main_cols if c not in raw_df.columns] # B: columns only in new data

print(f"A. Leakage columns NOT in main data ({len(missing_leak)}): {missing_leak}")
print(f"B. Columns ONLY in main data ({len(new_cols)}): {new_cols}")

- **7 leakage columns** (all settlement columns) do not exist in the main data, so **29 leakage columns** remain to drop.
- **2 new columns** (`open_il_6m`, `sec_app_open_il_6m`) count installment loans opened in the 6 months **before** applying.
  This comes from the credit report at application time, so they are **safe**.

## Step 19: Distribution of the target (`loan_status`) in the main data

In [0]:
# Count loans per status in the main table, with the % of all loans
display(spark.sql(f"""
    SELECT loan_status,
           COUNT(*) AS loans,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {RAW_TABLE}
    GROUP BY loan_status
    ORDER BY loans DESC
"""))

Databricks visualization. Run in Databricks to view.

"Current" is now **~53%** (vs 93.65% in the 2018 Q2 file). Among **finished** loans:
**522,325 Fully Paid (~79.4%)** vs **135,724 Charged Off/Default (~20.6%)**.

**Class imbalance:** a lazy model that always predicts "repaid" would already be ~79% accurate, yet catch **zero** defaults.
So accuracy is misleading here. We will evaluate with AUC-ROC, Precision, Recall and F1 instead.

## Step 20: Finished loans with a 0/1 default flag (EDA helper view)

In [0]:
# Temporary view: only finished loans, plus is_default (1 = Charged Off/Default, 0 = Fully Paid)
# The average of a 0/1 column = the default rate, which makes comparisons easy
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW finished_loans AS
    SELECT *,
           CASE WHEN loan_status IN ('Charged Off', 'Default') THEN 1 ELSE 0 END AS is_default
    FROM {RAW_TABLE}
    WHERE loan_status IN ('Fully Paid', 'Charged Off', 'Default')
""")

# Quick check: how many finished loans, how many defaults, and the default rate
display(spark.sql("""
    SELECT COUNT(*)                        AS finished_loans,
           SUM(is_default)                 AS defaults,
           ROUND(100 * AVG(is_default), 2) AS default_rate_pct
    FROM finished_loans
"""))

`finished_loans` view: **658,049** finished loans, **135,724** defaults, overall default rate **20.63%**.
This is the baseline every group below is compared against.

## Step 21: Default rate by loan grade

In [0]:
# Default rate per grade (A = safest ... G = riskiest), using the finished_loans view
display(spark.sql("""
    SELECT grade,
           COUNT(*)                        AS loans,
           ROUND(100 * AVG(is_default), 2) AS default_rate_pct
    FROM finished_loans
    GROUP BY grade
    ORDER BY grade
"""))

Databricks visualization. Run in Databricks to view.

Default rate rises steadily with grade: **A ≈ 6.5%** → B ≈ 13% → C ≈ 22% → D ≈ 30% → E ≈ 39% → F ≈ 45% → **G ≈ 50%**.
A grade G loan is about **7× more likely** to default than grade A, so `grade` is a strong predictor.
It is assigned at application review (before the loan starts), so it is **not** leakage.

## Step 22: Default rate by loan term (36 vs 60 months)

In [0]:
# Default rate per loan term (36 vs 60 months), using the finished_loans view
display(spark.sql("""
    SELECT term,
           COUNT(*)                        AS loans,
           ROUND(100 * AVG(is_default), 2) AS default_rate_pct
    FROM finished_loans
    GROUP BY term
    ORDER BY term
"""))

Databricks visualization. Run in Databricks to view.

60-month loans default about **twice as often** as 36-month loans (**~33.5% vs ~16.5%**).
Longer loans give more time for financial trouble, and borrowers who choose 60 months often need smaller monthly payments.
`term` is chosen on the application form, so it is **not** leakage, and it is a useful predictor.

## Step 23: Check whether a FICO score column exists (needed for the baseline rule)

In [0]:
# The brief's baseline rule is "reject if FICO < 650", so we need a FICO column.
# Search every column name for "fico" (lower-case, so "FICO" would also match)
fico_cols = [c for c in main_cols if "fico" in c.lower()]

print(f"FICO columns found ({len(fico_cols)}): {fico_cols}")

**No FICO column exists** in this dataset, so the brief's baseline rule ("reject if FICO < 650") cannot be applied directly.

**Plan B:** use Lending Club's `grade` as a proxy. Grade is the platform's own risk rating, built largely from credit history.
The baseline becomes an equally simple one-column rule, e.g. **"reject grades E, F, G"**. Several cutoffs will be compared in evaluation.
This keeps the spirit of the brief: our model must beat a blunt single-rule system.

## Step 24: Choose the model's columns
We keep **20 columns** that are known on application day and easy to explain:
12 numeric, 4 numbers-stored-as-text (`term`, `int_rate`, `emp_length`, `revol_util`), and 4 categorical
(`grade`, `home_ownership`, `verification_status`, `purpose`). The lists live in `00_config`.

**Left out:** leakage columns, IDs and free text, `sub_grade` (duplicate of grade), `funded_amnt*` (copies of loan_amnt),
mostly-empty joint/second-applicant columns, and location (`addr_state`, `zip_code`) to avoid unfair treatment by location.

## Step 25: Data quality report, Part 1 (type and missing values per column)

In [0]:
# For each model column: its type, how many values are missing, and the missing %
# (checked on finished loans, the rows the model will learn from)
df = spark.table("finished_loans")
total = df.count()

# Count nulls for ALL columns in one pass: isNull() → True/False → cast to 1/0 → sum
null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ALL_FEATURES]).first()

types = dict(df.dtypes)
report = [(c, types[c], null_counts[c], round(100 * null_counts[c] / total, 2)) for c in ALL_FEATURES]

display(spark.createDataFrame(report, ["column", "type", "nulls", "null_pct"]))

**Missing values:** 16 of 20 columns are complete. `mort_acc` has the most nulls (**7,495, 1.14%**),
then `revol_util` (368), `dti` (16) and `inq_last_6mths` (1). These will be filled in feature engineering.

**Types:** as expected, `term`, `int_rate`, `emp_length`, `revol_util` and the 4 categorical columns are text.
**Surprise:** `inq_last_6mths` is also text, although it should be a number. Investigated in Part 2.
**Watch out:** `emp_length` shows 0 nulls, but missing values may be stored as the text "n/a".

## Step 26: Data quality report, Part 2a (ranges of the number columns)

In [0]:
# Min, median and max for each number column, to spot impossible or extreme values
# (inq_last_6mths is skipped here because it is text; it gets checked with the text columns)
number_cols = [c for c in NUMERIC_FEATURES if c != "inq_last_6mths"]

display(df.select(number_cols).summary("min", "50%", "max"))

In [0]:
# How many loans have impossible or suspicious values?
display(df.select(
    F.sum((F.col("dti") < 0).cast("int")).alias("dti_negative"),
    F.sum((F.col("dti") > 100).cast("int")).alias("dti_over_100"),
    F.sum((F.col("annual_inc") == 0).cast("int")).alias("income_zero"),
))

**Ranges:** most columns look normal (e.g. `loan_amnt` 1,000–40,000, median income $65,000).
**Impossible values:** `dti` has 1 negative value and 36 values above 100 (max 999, likely a code for "unknown"),
and 16 loans have an income of 0 (likely the same loans whose `dti` is empty, since debt ÷ 0 cannot be calculated).
Only ~53 loans in total → they will be treated as **missing** and filled in feature engineering.
**Extreme but possible** values (e.g. an income of $8.9M, 86 public records) are kept for now.

## Step 27: Data quality report, Part 2b (values of the text columns)

In [0]:
# For each text column: how many different values it has, and up to 15 examples
text_cols = TEXT_NUMBER_FEATURES + CATEGORICAL_FEATURES + ["inq_last_6mths"]

for c in text_cols:
    n_distinct = df.select(c).distinct().count()
    examples = [row[0] for row in df.select(c).distinct().limit(15).collect()]
    print(f"{c} ({n_distinct} distinct): {examples}\n")

In [0]:
# How many emp_length values are the hidden missing "n/a"?
# And how rare are the odd home_ownership values (ANY, NONE, OTHER)?
display(df.groupBy("emp_length").count().filter(F.col("emp_length") == "n/a"))
display(df.groupBy("home_ownership").count().orderBy(F.desc("count")))

**Text columns:**
- `term`, `int_rate`, `revol_util` are numbers stored as text, with **hidden leading spaces** and units ("months", "%"). Clean and convert.
- `inq_last_6mths` only contains "0"–"8" in finished loans, so it just needs converting to a number.
- `emp_length` hides missing values as the text **"n/a" (33,672 loans, ~5.1%)**, the most-missing column overall.
  Convert "< 1 year" → 0 ... "10+ years" → 10, treat "n/a" as missing, and consider a "was missing" flag.
- `home_ownership`: ANY (46), OTHER (45) and NONE (40) are very rare (131 loans) → merge into **OTHER**.
- `grade` (7), `verification_status` (3) and `purpose` (14) have small, clean sets of values → ready to encode.

# EDA Summary

**Data:** Lending Club 2012–2017 (1,481,541 valid loans; 658,049 with a final outcome). The suggested 2018 Q2 file was rejected (93.65% unfinished loans).

**Target:** 79.4% Fully Paid vs **20.6% Default** (finished loans) → imbalanced, so accuracy is not a valid main metric.

**Leakage:** 29 columns created *after* the loan starts (payments, recoveries, hardship) → dropped (`LEAKAGE_COLS` in `00_config`).

**Key relationships:** default rate rises from ~6.5% (grade A) to ~50% (grade G), and 60-month loans default ~2× more than 36-month loans.

**Baseline:** no FICO column exists → the "FICO < 650" rule is replaced by an equivalent one-column rule on `grade`.

**Decisions for feature engineering:**
| Issue | Columns | Plan |
|---|---|---|
| Numbers stored as text | `term`, `int_rate`, `revol_util`, `emp_length`, `inq_last_6mths` | Remove spaces/units and convert |
| Hidden missing values | `emp_length` = "n/a" (5.1%) | Treat as missing (+ consider a flag) |
| Impossible values | `dti` < 0 or > 100, `annual_inc` = 0 (~53 loans) | Treat as missing |
| Missing values | `mort_acc` (1.14%), `revol_util`, `dti`, `inq_last_6mths` | Fill (strategy chosen in Step 2) |
| Rare categories | `home_ownership`: ANY / NONE / OTHER (131 loans) | Merge into OTHER |
| Target | `loan_status` | Fully Paid → 0; Charged Off / Default → 1; decide on Current / Late |
| Split | `issue_d` (2012–2017) | Time-based train/test split |